# Bangla Hallucination Detection — v7.0

**Goal of this revision: fix the "only 299 rows" problem the right way — task-shaped synthetic pretraining from real Bangla QA datasets, plus two extra independent signals (a second NLI checkpoint and an LLM verifier) — while keeping the honest nested-CV evaluation from v6 intact.**

### What changed vs v6.0

1. **Synthetic pretraining corpus (new, Section 1b).** Built from `sartajekram/BanglaRQA` and
   `rasheduzzaman/Bangla_question_answer_pair_70K_dataset` — both are `(context, question, answer)`
   triples, the *right shape* for this task, unlike generic instruction datasets (Alpaca-Bangla etc.),
   which have no faithfulness label and were deliberately **not** used. Each real QA triple becomes a
   `label=1` (faithful) example; each is paired with a synthetically corrupted `label=0` (hallucinated)
   example via four corruption strategies (cross-context answer swap, number/entity substitution,
   negation, prompt-only answer with no grounding). This gives ~6–8k *task-shaped* rows instead of raw
   Bangla text that the model would gain nothing task-specific from.
2. **Two-stage BanglaBERT training.** Stage 1: pretrain `csebuetnlp/banglabert` for 2 epochs on the
   synthetic corpus (domain/task adaptation). Stage 2: the existing 5-fold fine-tune on your real 299
   rows, now initialized from the Stage-1 checkpoint instead of the raw pretrained weights. This is the
   standard low-data trick: adapt to the task shape on synthetic data, then specialize on the small real
   labeled set.
3. **Added a second, independent NLI checkpoint** (`cross-encoder/nli-deberta-v3-base` family via
   `MoritzLaurer` alt / `cross-encoder`) for entailment/contradiction — diversifies the NLI signal beyond
   a single model's blind spots. Kept the original mDeBERTa NLI too.
4. **Added an optional small-LLM verifier feature** (`Qwen2.5-1.5B-Instruct`, 4-bit) that directly answers
   "is this response supported by this context?" per row — run only on your 299 train + test rows (not on
   any large corpus), so it stays cheap. Gated by `ENABLE_QWEN` since it's the most time-expensive new
   piece; turn off first if you're tight on the 9-hour Kaggle budget.
5. **Honest nested-CV evaluation kept unchanged from v6** (Section 10) — now scored over 7 blend
   components instead of 5. Section 10's number is still the one to trust, not Section 11.

### What was deliberately NOT used, and why
`alpaca_bangla`, `alpaca-cleaned-bengali`, `bengali_alpaca_dolly_67k`, the NER/financial-news/aggressive-
text/money datasets — none of these carry a faithfulness label or a context+response pair shaped like
this task. Concatenating them in would add noise, not signal. "More data" only helps when it's shaped
like the thing you're predicting.

### Time budget on Kaggle (T4/P100, 9h session cap)
- Synthetic corpus build: CPU string work, a few minutes once datasets are downloaded.
- Stage-1 BanglaBERT pretrain (2 epochs, ~6-8k rows): ~30–45 min on a T4.
- Stage-2 5-fold fine-tune on 299 rows: minutes (unchanged from v6).
- Second NLI + embeddings + NER: unchanged from v6, ~10-20 min total.
- Qwen verifier (optional, 299 train + test rows only): ~15-30 min depending on test set size.
- **Total: comfortably under 9h with margin**, even with everything enabled.


## 0. Setup

In [ ]:
!pip install -q transformers sentence-transformers xgboost accelerate datasets bitsandbytes
!pip install -q git+https://github.com/csebuetnlp/normalizer

In [ ]:
import json, gc, os, re, warnings, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
                           get_linear_schedule_with_warmup)
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SKPipeline
import xgboost as xgb
from tqdm.auto import tqdm
from normalizer import normalize
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

DATA_PATH = "/kaggle/input/competitions/bengali-hallucination/dataset samples.json"
TEST_PATH = "/kaggle/input/competitions/bengali-hallucination/test set.csv"
SUBMISSION_PATH = "/kaggle/working/submission_v7.0.csv"

NLI_MODEL = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
NLI_MODEL_2 = "cross-encoder/nli-deberta-v3-base"     # second, independent NLI checkpoint (English-trained
                                                        # but works cross-lingually via mDeBERTa-style XLM-R
                                                        # backbones reasonably well on translated/short claims;
                                                        # swap for another multilingual NLI ckpt if you find
                                                        # one that scores higher on your OOF)
EMB_MODEL = "intfloat/multilingual-e5-base"
NER_MODEL = "Davlan/xlm-roberta-base-ner-hrl"
BANGLABERT_MODEL = "csebuetnlp/banglabert"
QWEN_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

# Synthetic pretraining corpus sources (real Bangla QA triples, right shape for this task)
SYN_SOURCES = [
    {"hf_name": "sartajekram/BanglaRQA", "config": None},
    {"hf_name": "rasheduzzaman/Bangla_question_answer_pair_70K_dataset", "config": None},
]
SYN_TARGET_ROWS = 6000     # balanced target size (half faithful, half corrupted) for Stage-1 pretrain
SYN_PRETRAIN_EPOCHS = 2
BANGLABERT_PRETRAINED_PATH = "/kaggle/working/banglabert_stage1"

N_FOLDS = 5
MAX_LEN = 256

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cpu":
    print("WARNING: no GPU detected. BanglaBERT fine-tuning, synthetic pretraining, and the transformer "
          "feature extractors will be very slow on CPU. Disable ENABLE_BANGLABERT / ENABLE_SYN_PRETRAIN / "
          "ENABLE_QWEN / ENABLE_NER below if you need a quick run.")

ENABLE_BANGLABERT   = True   # main learned signal
ENABLE_SYN_PRETRAIN = True   # Stage-1 pretrain on synthetic corpus before fine-tuning on real 299 rows
ENABLE_NLI2         = True   # second NLI checkpoint
ENABLE_NER          = True   # slower, moderate signal
ENABLE_QWEN         = True   # LLM verifier feature — most expensive new addition, disable first if tight on time

## 1. Load & Clean Real Training Data

In [ ]:
if not os.path.exists(DATA_PATH):
    DATA_PATH = "../bengali-hallucination/dataset samples.json"
    TEST_PATH = "../bengali-hallucination/test set.csv"

with open(DATA_PATH, encoding='utf-8') as f:
    records = json.load(f)
df = pd.DataFrame(records)
print(f"Train: {len(df)}")

NO_CTX = {"", "nan", "NaN", "[NULL]", None}
def clean(text):
    if pd.isna(text) or str(text).strip() in NO_CTX:
        return ""
    return normalize(str(text).strip())

for col in ['context', 'prompt_bn', 'response_bn']:
    df[col] = df[col].apply(clean)

df['has_context'] = df['context'].str.len() > 0
df['premise'] = df['context'].where(df['has_context'], df['prompt_bn'])

print(f"With context: {df['has_context'].sum()}, Without: {(~df['has_context']).sum()}")
print(f"Label 0 (hallucinated): {(df['label']==0).sum()}, Label 1 (faithful): {(df['label']==1).sum()}")

# Fixed 5-fold split reused everywhere below so every OOF array lines up on the same indices
skf = StratifiedKFold(N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
y = df['label'].values
fold_splits = list(skf.split(df, y))

## 1b. Synthetic Task-Shaped Pretraining Corpus (new in v7)

Real Bangla `(context, question, answer)` triples from BanglaRQA and the rasheduzzaman QA-pair dataset
give correct answers only — that's a `label=1` (faithful) example as-is. To get `label=0` (hallucinated)
examples in the *same distribution*, each real triple is corrupted with one of four strategies, chosen at
random per row so the corpus doesn't overfit to a single corruption pattern:

1. **Cross-context swap** — replace the answer with the answer from a different, randomly chosen row's
   context (classic "right-shaped, wrong-source" hallucination).
2. **Number/entity substitution** — if the answer contains a number, perturb it; otherwise swap out a
   capitalized/quoted token-like span for another row's token.
3. **Negation** — prepend/insert a negation cue so the claim contradicts the context.
4. **No-grounding answer** — keep the question but attach an answer drawn only from the *question* of a
   different row, with no relation to the paired context at all (tests the "not grounded" pattern
   distinct from "wrong-but-related").

This mirrors the shape of your real 299-row set (mix of context / no-context rows, both label classes)
without inventing a task-irrelevant corpus. If you find `honest_cv_score` in Section 10 doesn't improve
after adding this, that's a real signal the synthetic distribution doesn't match the true test set well —
don't just keep it because more data was added; check the honest score, not intuition.

In [ ]:
from datasets import load_dataset

def load_syn_qa_triples(max_per_source=15000):
    """Load (context, question, answer) triples from the configured HF sources.
    Falls back gracefully (prints a warning, returns []) if a dataset is unreachable —
    the rest of the pipeline still runs on whatever sources succeeded."""
    triples = []
    for src in SYN_SOURCES:
        try:
            ds = load_dataset(src["hf_name"], src["config"]) if src["config"] else load_dataset(src["hf_name"])
            split = ds["train"] if "train" in ds else list(ds.values())[0]
            n_taken = 0
            for row in split:
                ctx = row.get("context") or row.get("passage") or row.get("paragraph") or ""
                q = row.get("question") or row.get("prompt") or row.get("question_text") or ""
                a = row.get("answer") or row.get("answer_text") or row.get("answers") or ""
                if isinstance(a, dict):
                    a = a.get("text", [""])
                    a = a[0] if isinstance(a, list) and a else ""
                if isinstance(a, list):
                    a = a[0] if a else ""
                ctx, q, a = str(ctx).strip(), str(q).strip(), str(a).strip()
                if not q or not a:
                    continue
                triples.append({"context": ctx, "question": q, "answer": a, "source": src["hf_name"]})
                n_taken += 1
                if n_taken >= max_per_source:
                    break
            print(f"  loaded {n_taken} triples from {src['hf_name']}")
        except Exception as e:
            print(f"  WARNING: could not load {src['hf_name']} ({e}). Skipping this source.")
    return triples

print("Loading synthetic pretraining sources...")
syn_triples = load_syn_qa_triples(max_per_source=SYN_TARGET_ROWS) if ENABLE_SYN_PRETRAIN else []
print(f"Total raw QA triples available: {len(syn_triples)}")

In [ ]:
BN_NUM_RE = re.compile(r'[০-৯0-9]+')

def corrupt_number(text):
    def bump(m):
        s = m.group(0)
        try:
            v = int(s)
            return str(v + random.choice([-10, -5, -2, -1, 1, 2, 5, 10]))
        except Exception:
            return s
    new_text, n = BN_NUM_RE.subn(bump, text, count=1)
    return new_text if n > 0 else None

NEGATION_CUES = ["না, ", "কখনোই না — ", "এটি সত্য নয় যে "]

def build_synthetic_dataset(triples, target_rows=SYN_TARGET_ROWS, seed=RANDOM_STATE):
    rng = random.Random(seed)
    if not triples:
        return pd.DataFrame(columns=['context', 'prompt_bn', 'response_bn', 'label', 'has_context', 'premise'])

    n_pairs = min(target_rows // 2, len(triples))
    chosen = rng.sample(triples, n_pairs)
    rows = []
    for t in chosen:
        ctx, q, a = t['context'], t['question'], t['answer']
        # positive (faithful) example — the real answer
        rows.append({'context': ctx, 'prompt_bn': q, 'response_bn': a, 'label': 1})

        # pick a different row for corruption material
        other = rng.choice(triples)
        while other is t and len(triples) > 1:
            other = rng.choice(triples)

        strategy = rng.choice(['cross_context', 'number_swap', 'negation', 'no_grounding'])
        if strategy == 'cross_context':
            bad_answer = other['answer']
        elif strategy == 'number_swap':
            bumped = corrupt_number(a)
            bad_answer = bumped if bumped else other['answer']
        elif strategy == 'negation':
            cue = rng.choice(NEGATION_CUES)
            bad_answer = cue + a
        else:  # no_grounding
            bad_answer = other['question']

        rows.append({'context': ctx, 'prompt_bn': q, 'response_bn': bad_answer, 'label': 0})

    syn_df = pd.DataFrame(rows)
    for col in ['context', 'prompt_bn', 'response_bn']:
        syn_df[col] = syn_df[col].apply(clean)
    syn_df['has_context'] = syn_df['context'].str.len() > 0
    syn_df['premise'] = syn_df['context'].where(syn_df['has_context'], syn_df['prompt_bn'])
    syn_df = syn_df[(syn_df['prompt_bn'].str.len() > 0) & (syn_df['response_bn'].str.len() > 0)].reset_index(drop=True)
    return syn_df

syn_df = build_synthetic_dataset(syn_triples) if ENABLE_SYN_PRETRAIN else pd.DataFrame()
print(f"Synthetic Stage-1 corpus: {len(syn_df)} rows "
      f"(label 0={ (syn_df['label']==0).sum() if len(syn_df) else 0 }, "
      f"label 1={ (syn_df['label']==1).sum() if len(syn_df) else 0 })")

## 2. Fine-tuned BanglaBERT Classifier (two-stage in v7)

In [ ]:
class PairDataset(Dataset):
    def __init__(self, premises, responses, labels, tokenizer, max_len=MAX_LEN):
        self.premises = list(premises)
        self.responses = list(responses)
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.premises)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.premises[idx], self.responses[idx],
            truncation=True, max_length=self.max_len, padding='max_length',
            return_tensors='pt'
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def train_one_fold(train_df, val_df, init_model_path=BANGLABERT_MODEL,
                    epochs=6, batch_size=8, lr=2e-5, weight_decay=0.01, patience=2):
    tokenizer = AutoTokenizer.from_pretrained(BANGLABERT_MODEL)
    model = AutoModelForSequenceClassification.from_pretrained(init_model_path, num_labels=2).to(device)

    n0 = (train_df['label'] == 0).sum()
    n1 = (train_df['label'] == 1).sum()
    class_weights = torch.tensor([len(train_df) / (2 * max(n0, 1)),
                                   len(train_df) / (2 * max(n1, 1))], dtype=torch.float).to(device)
    loss_fn = nn.CrossEntropyLoss(weight=class_weights)

    train_ds = PairDataset(train_df['premise'], train_df['response_bn'], train_df['label'].values, tokenizer)
    val_ds = PairDataset(val_df['premise'], val_df['response_bn'], val_df['label'].values, tokenizer)
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=batch_size * 2, shuffle=False)

    optim = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    total_steps = len(train_dl) * epochs
    sched = get_linear_schedule_with_warmup(optim, num_warmup_steps=int(0.1 * total_steps),
                                             num_training_steps=total_steps)

    best_f1, best_state, no_improve = -1, None, 0
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for batch in train_dl:
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop('labels')
            optim.zero_grad()
            out = model(**batch)
            loss = loss_fn(out.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()
            sched.step()
            train_loss += loss.item()

        model.eval()
        val_probs, val_labels = [], []
        with torch.no_grad():
            for batch in val_dl:
                batch = {k: v.to(device) for k, v in batch.items()}
                labels = batch.pop('labels')
                out = model(**batch)
                probs = F.softmax(out.logits, dim=-1)[:, 1]
                val_probs.extend(probs.cpu().numpy().tolist())
                val_labels.extend(labels.cpu().numpy().tolist())
        val_preds = (np.array(val_probs) >= 0.5).astype(int)
        val_f1 = f1_score(val_labels, val_preds, average='macro')
        print(f"    epoch {epoch+1}/{epochs}  train_loss={train_loss/len(train_dl):.4f}  val_macroF1={val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print("    early stopping")
                break

    model.load_state_dict(best_state)
    return model, tokenizer, best_f1

### Stage 1 — pretrain on the synthetic corpus (new in v7)

Trained once on the full synthetic set (no CV needed here — this is a warm start, not the scored model),
then saved to disk. Every one of the 5 real-data folds in Stage 2 below initializes from this checkpoint
instead of the raw `csebuetnlp/banglabert` weights.

In [ ]:
if ENABLE_BANGLABERT and ENABLE_SYN_PRETRAIN and len(syn_df) > 0:
    print(f"Stage 1: pretraining BanglaBERT on {len(syn_df)} synthetic rows for {SYN_PRETRAIN_EPOCHS} epochs...")
    syn_tokenizer = AutoTokenizer.from_pretrained(BANGLABERT_MODEL)
    syn_model = AutoModelForSequenceClassification.from_pretrained(BANGLABERT_MODEL, num_labels=2).to(device)

    syn_ds = PairDataset(syn_df['premise'], syn_df['response_bn'], syn_df['label'].values, syn_tokenizer)
    syn_dl = DataLoader(syn_ds, batch_size=16, shuffle=True)

    syn_optim = AdamW(syn_model.parameters(), lr=2e-5, weight_decay=0.01)
    syn_total_steps = len(syn_dl) * SYN_PRETRAIN_EPOCHS
    syn_sched = get_linear_schedule_with_warmup(syn_optim, num_warmup_steps=int(0.1 * syn_total_steps),
                                                 num_training_steps=syn_total_steps)
    syn_loss_fn = nn.CrossEntropyLoss()

    syn_model.train()
    for epoch in range(SYN_PRETRAIN_EPOCHS):
        total_loss = 0.0
        for batch in tqdm(syn_dl, desc=f"Stage-1 epoch {epoch+1}/{SYN_PRETRAIN_EPOCHS}"):
            batch = {k: v.to(device) for k, v in batch.items()}
            labels = batch.pop('labels')
            syn_optim.zero_grad()
            out = syn_model(**batch)
            loss = syn_loss_fn(out.logits, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(syn_model.parameters(), 1.0)
            syn_optim.step()
            syn_sched.step()
            total_loss += loss.item()
        print(f"  epoch {epoch+1}: train_loss={total_loss/len(syn_dl):.4f}")

    os.makedirs(BANGLABERT_PRETRAINED_PATH, exist_ok=True)
    syn_model.save_pretrained(BANGLABERT_PRETRAINED_PATH)
    syn_tokenizer.save_pretrained(BANGLABERT_PRETRAINED_PATH)
    BANGLABERT_INIT_PATH = BANGLABERT_PRETRAINED_PATH
    del syn_model
    gc.collect(); torch.cuda.empty_cache() if device == 'cuda' else None
    print(f"Stage-1 checkpoint saved to {BANGLABERT_PRETRAINED_PATH}")
else:
    BANGLABERT_INIT_PATH = BANGLABERT_MODEL
    print("Synthetic pretraining disabled or no synthetic rows available — "
          "Stage 2 will initialize directly from the raw pretrained checkpoint (same as v6).")

### Stage 2 — 5-fold fine-tune on the real 299 rows (same procedure as v6, now warm-started)

In [ ]:
banglabert_oof = np.full(len(df), 0.5)
banglabert_fold_models = []  # kept for test-time ensembling

if ENABLE_BANGLABERT:
    for fold, (tr_idx, va_idx) in enumerate(fold_splits):
        print(f"\n--- BanglaBERT fold {fold} ---")
        tr_df, va_df = df.iloc[tr_idx], df.iloc[va_idx]
        model, tokenizer, fold_f1 = train_one_fold(tr_df, va_df, init_model_path=BANGLABERT_INIT_PATH)

        model.eval()
        val_ds = PairDataset(va_df['premise'], va_df['response_bn'], None, tokenizer)
        val_dl = DataLoader(val_ds, batch_size=16, shuffle=False)
        probs = []
        with torch.no_grad():
            for batch in val_dl:
                batch = {k: v.to(device) for k, v in batch.items()}
                out = model(**batch)
                probs.extend(F.softmax(out.logits, dim=-1)[:, 1].cpu().numpy().tolist())
        banglabert_oof[va_idx] = probs
        banglabert_fold_models.append((model, tokenizer))
        del model
        gc.collect(); torch.cuda.empty_cache() if device == 'cuda' else None

    df['banglabert_oof'] = banglabert_oof
    oof_acc = accuracy_score(y, (banglabert_oof >= 0.5).astype(int))
    oof_f1 = f1_score(y, (banglabert_oof >= 0.5).astype(int), average='macro')
    print(f"\n=== BanglaBERT OOF: Acc={oof_acc:.4f}  MacroF1={oof_f1:.4f} ===")
else:
    df['banglabert_oof'] = 0.5
    print("BanglaBERT disabled — feature set to neutral 0.5 for all rows.")

## 3. Hand-Crafted Features (unchanged from v5/v6)

In [ ]:
BN_DIGITS = {'০':0,'১':1,'২':2,'৩':3,'৪':4,'৫':5,'৬':6,'৭':7,'৮':8,'৯':9}

def bn_to_int(s):
    try:
        return int(s)
    except Exception:
        pass
    try:
        s2 = s
        for bn, ar in BN_DIGITS.items():
            s2 = s2.replace(bn, str(ar))
        return int(s2)
    except Exception:
        return None

def extract_numbers(text):
    nums = re.findall(r'[০-৯]+|[0-9]+(?:\.[0-9]+)?', text)
    result = []
    for n in nums:
        v = bn_to_int(n)
        if v is not None:
            result.append(v)
    return result

def tokenize_bangla(text):
    return [w for w in re.split(r'[\s,\.\?\!\?\"\'\:\;\(\)\[\]\{\}]+', text) if len(w) > 0]

def jaccard(a, b):
    if not a or not b:
        return 0.0
    sa, sb = set(a), set(b)
    return len(sa & sb) / max(len(sa | sb), 1)

def common_ratio(a, b):
    if not b:
        return 0.0
    return len(set(a) & set(b)) / max(len(set(b)), 1)

def build_features(data_df):
    feats = pd.DataFrame(index=data_df.index)
    feats['ctx_len'] = data_df['context'].str.len()
    feats['prompt_len'] = data_df['prompt_bn'].str.len()
    feats['resp_len'] = data_df['response_bn'].str.len()
    feats['has_context'] = data_df['has_context'].astype(int)
    feats['resp_prompt_ratio'] = feats['resp_len'] / (feats['prompt_len'] + 1)
    feats['resp_ctx_ratio'] = feats['resp_len'] / (feats['ctx_len'] + 1)
    feats['ctx_prompt_ratio'] = feats['ctx_len'] / (feats['prompt_len'] + 1)

    prompt_toks = data_df['prompt_bn'].apply(tokenize_bangla)
    resp_toks = data_df['response_bn'].apply(tokenize_bangla)
    ctx_toks = data_df['context'].apply(tokenize_bangla)

    feats['prompt_n_tokens'] = prompt_toks.apply(len)
    feats['resp_n_tokens'] = resp_toks.apply(len)
    feats['ctx_n_tokens'] = ctx_toks.apply(len)

    feats['resp_in_ctx_jaccard'] = [jaccard(r, c) for r, c in zip(resp_toks, ctx_toks)]
    feats['resp_in_ctx_ratio'] = [common_ratio(r, c) for r, c in zip(resp_toks, ctx_toks)]
    feats['ctx_in_resp_ratio'] = [common_ratio(c, r) for r, c in zip(resp_toks, ctx_toks)]
    feats['resp_prompt_jaccard'] = [jaccard(r, p) for r, p in zip(resp_toks, prompt_toks)]
    feats['resp_in_prompt_ratio'] = [common_ratio(r, p) for r, p in zip(resp_toks, prompt_toks)]

    ctx_nums = data_df['context'].apply(extract_numbers)
    resp_nums = data_df['response_bn'].apply(extract_numbers)
    prompt_nums = data_df['prompt_bn'].apply(extract_numbers)

    feats['resp_has_number'] = resp_nums.apply(len).clip(upper=1)
    feats['ctx_has_number'] = ctx_nums.apply(len).clip(upper=1)
    feats['prompt_has_number'] = prompt_nums.apply(len).clip(upper=1)

    def numbers_match(rn, cn):
        if not rn:
            return 0
        return int(any(n in cn for n in rn))

    feats['resp_num_in_ctx'] = [numbers_match(rn, cn) for rn, cn in zip(resp_nums, ctx_nums)]
    feats['resp_num_count'] = resp_nums.apply(len)
    feats['ctx_num_count'] = ctx_nums.apply(len)
    feats['prompt_num_count'] = prompt_nums.apply(len)

    feats['resp_very_short'] = (feats['resp_len'] < 15).astype(int)
    feats['resp_one_word'] = (feats['resp_n_tokens'] <= 2).astype(int)

    feats['resp_starts_with_ctx'] = [
        int(str(r).startswith(str(c)[:20])) if c and r else 0
        for r, c in zip(data_df['response_bn'], data_df['context'])
    ]

    feats['resp_extrinsic_words'] = [len(set(r) - set(c)) if c else 0 for r, c in zip(resp_toks, ctx_toks)]
    feats['resp_extrinsic_ratio'] = [
        len(set(r) - set(c)) / max(len(set(r)), 1) if c else 1.0 for r, c in zip(resp_toks, ctx_toks)
    ]
    feats['ctx_resp_shared_entities'] = [len(set(r) & set(c)) if c else 0 for r, c in zip(resp_toks, ctx_toks)]
    feats['ctx_resp_entity_overlap'] = [
        len(set(r) & set(c)) / max(len(set(r) | set(c)), 1) if c else 0 for r, c in zip(resp_toks, ctx_toks)
    ]
    feats['prompt_resp_shared_entities'] = [len(set(p) & set(r)) for p, r in zip(prompt_toks, resp_toks)]
    feats['prompt_resp_entity_overlap'] = [
        len(set(p) & set(r)) / max(len(set(p) | set(r)), 1) for p, r in zip(prompt_toks, resp_toks)
    ]
    feats['ctx_completeness'] = [
        len(set(c) & set(p)) / max(len(set(p)), 1) if c else 0 for c, p in zip(ctx_toks, prompt_toks)
    ]
    return feats.fillna(0)

print("Building hand-crafted features...")
hc_train = build_features(df)
print(f"Hand-crafted features: {hc_train.shape[1]} columns")

## 4. Primary NLI Features (mDeBERTa, unchanged from v6)

In [ ]:
nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(device)
nli_model.eval()
# MoritzLaurer's checkpoint label order is entailment=0, neutral=1, contradiction=2
NLI_ENTAIL_IDX, NLI_NEUTRAL_IDX, NLI_CONTRA_IDX = 0, 1, 2

@torch.no_grad()
def get_nli_scores(data_df, tokenizer, model, entail_idx=0, neutral_idx=1, contra_idx=2, batch_size=16):
    premises = data_df['premise'].tolist()
    hypotheses = data_df['response_bn'].tolist()
    entail = np.full(len(data_df), 1/3)
    neutral = np.full(len(data_df), 1/3)
    contra = np.full(len(data_df), 1/3)

    for start in tqdm(range(0, len(data_df), batch_size), desc='NLI'):
        end = start + batch_size
        p_batch = premises[start:end]
        h_batch = hypotheses[start:end]
        valid = [i for i, (p, h) in enumerate(zip(p_batch, h_batch)) if p and h]
        if not valid:
            continue
        enc = tokenizer([p_batch[i] for i in valid], [h_batch[i] for i in valid],
                         truncation=True, max_length=384, padding=True, return_tensors='pt').to(device)
        logits = model(**enc).logits
        probs = F.softmax(logits, dim=-1).cpu().numpy()
        for j, i in enumerate(valid):
            entail[start + i] = probs[j, entail_idx]
            neutral[start + i] = probs[j, neutral_idx]
            contra[start + i] = probs[j, contra_idx]
    return entail, neutral, contra

print("Computing primary NLI scores for training data...")
nli_entail, nli_neutral, nli_contra = get_nli_scores(df, nli_tokenizer, nli_model,
                                                      NLI_ENTAIL_IDX, NLI_NEUTRAL_IDX, NLI_CONTRA_IDX)
df['nli_entail'] = nli_entail
df['nli_neutral'] = nli_neutral
df['nli_contra'] = nli_contra
print(f"NLI entail: mean={nli_entail.mean():.3f}  neutral: mean={nli_neutral.mean():.3f}  contra: mean={nli_contra.mean():.3f}")

ctx_mask = df['has_context']
ctx_acc = accuracy_score(df.loc[ctx_mask, 'label'], (nli_entail[ctx_mask] > 0.5).astype(int))
print(f"NLI-entailment-only accuracy on context rows: {ctx_acc:.4f}")

del nli_model
gc.collect(); torch.cuda.empty_cache() if device == 'cuda' else None

## 4b. Second, Independent NLI Checkpoint (new in v7)

A different NLI model trained on different data picks up different failure modes than the mDeBERTa
checkpoint above — the two are combined as separate blend components later, not averaged together, so
the blend search can decide how much to trust each on your actual labels.

In [ ]:
if ENABLE_NLI2:
    nli2_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_2)
    nli2_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_2).to(device)
    nli2_model.eval()
    # cross-encoder/nli-deberta-v3-base label order: contradiction=0, entailment=1, neutral=2
    NLI2_CONTRA_IDX, NLI2_ENTAIL_IDX, NLI2_NEUTRAL_IDX = 0, 1, 2

    print("Computing second NLI checkpoint scores for training data...")
    nli2_entail, nli2_neutral, nli2_contra = get_nli_scores(
        df, nli2_tokenizer, nli2_model, NLI2_ENTAIL_IDX, NLI2_NEUTRAL_IDX, NLI2_CONTRA_IDX
    )
    df['nli2_entail'] = nli2_entail
    df['nli2_contra'] = nli2_contra
    print(f"NLI-2 entail: mean={nli2_entail.mean():.3f}  contra: mean={nli2_contra.mean():.3f}")

    del nli2_model
    gc.collect(); torch.cuda.empty_cache() if device == 'cuda' else None
else:
    df['nli2_entail'] = 1/3
    df['nli2_contra'] = 1/3
    print("Second NLI checkpoint disabled — features set to neutral 1/3.")

## 5. Embedding Similarity (multilingual-e5-base, unchanged from v5/v6)

In [ ]:
emb_model = SentenceTransformer(EMB_MODEL, device=str(device))

def cosine_sim_matrix(texts_a, texts_b, batch_size=64):
    emb_a = emb_model.encode(texts_a, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)
    emb_b = emb_model.encode(texts_b, batch_size=batch_size, show_progress_bar=True, normalize_embeddings=True)
    sims = np.sum(emb_a * emb_b, axis=1)
    return sims, emb_a, emb_b

print("Encoding training texts...")
ctx_texts = df['premise'].tolist()
resp_texts = df['response_bn'].tolist()
prompt_texts = df['prompt_bn'].tolist()

sim_ctx_resp, emb_ctx, emb_resp = cosine_sim_matrix(ctx_texts, resp_texts)
sim_prompt_resp, _, _ = cosine_sim_matrix(prompt_texts, resp_texts)

df['sim_ctx_resp'] = sim_ctx_resp
df['sim_prompt_resp'] = sim_prompt_resp
print(f"sim_ctx_resp: mean={sim_ctx_resp.mean():.3f}  sim_prompt_resp: mean={sim_prompt_resp.mean():.3f}")

del emb_model
gc.collect(); torch.cuda.empty_cache() if device == 'cuda' else None

## 6. NER Features (unchanged from v5/v6, optional)

In [ ]:
def get_ner_features(data_df):
    ner_feats = pd.DataFrame(index=data_df.index)
    if not ENABLE_NER:
        for col in ['ctx_entity_count', 'resp_entity_count', 'prompt_entity_count',
                    'resp_ctx_entity_overlap', 'resp_prompt_entity_overlap',
                    'resp_entities_in_ctx', 'resp_entities_in_prompt',
                    'resp_extrinsic_entities', 'resp_extrinsic_entity_ratio']:
            ner_feats[col] = 0.0
        return ner_feats

    from transformers import pipeline as hf_pipeline
    ner_pipe = hf_pipeline("ner", model=NER_MODEL, device=0 if device == 'cuda' else -1,
                            aggregation_strategy="simple")

    def extract_entities(text):
        if not text or len(text.strip()) == 0:
            return []
        try:
            return [e['word'] for e in ner_pipe(text)]
        except Exception:
            return []

    ctx_entities = data_df['context'].apply(extract_entities)
    resp_entities = data_df['response_bn'].apply(extract_entities)
    prompt_entities = data_df['prompt_bn'].apply(extract_entities)

    ner_feats['ctx_entity_count'] = ctx_entities.apply(len)
    ner_feats['resp_entity_count'] = resp_entities.apply(len)
    ner_feats['prompt_entity_count'] = prompt_entities.apply(len)
    ner_feats['resp_ctx_entity_overlap'] = [
        len(set(r) & set(c)) / max(len(set(r) | set(c)), 1) if c else 0
        for r, c in zip(resp_entities, ctx_entities)
    ]
    ner_feats['resp_prompt_entity_overlap'] = [
        len(set(r) & set(p)) / max(len(set(r) | set(p)), 1)
        for r, p in zip(resp_entities, prompt_entities)
    ]
    ner_feats['resp_entities_in_ctx'] = [
        len(set(r) & set(c)) / max(len(set(r)), 1) if c else 0
        for r, c in zip(resp_entities, ctx_entities)
    ]
    ner_feats['resp_entities_in_prompt'] = [
        len(set(r) & set(p)) / max(len(set(r)), 1)
        for r, p in zip(resp_entities, prompt_entities)
    ]
    ner_feats['resp_extrinsic_entities'] = [
        len(set(r) - set(c)) if c else len(set(r)) for r, c in zip(resp_entities, ctx_entities)
    ]
    ner_feats['resp_extrinsic_entity_ratio'] = [
        len(set(r) - set(c)) / max(len(set(r)), 1) if c else 1.0
        for r, c in zip(resp_entities, ctx_entities)
    ]
    del ner_pipe
    gc.collect(); torch.cuda.empty_cache() if device == 'cuda' else None
    return ner_feats.fillna(0)

print("Computing NER features for training data...")
ner_train = get_ner_features(df)
print(f"NER features: {ner_train.shape[1]} columns")

## 6b. LLM Verifier Feature (new in v7, optional — `ENABLE_QWEN`)

Runs a small instruct model (`Qwen2.5-1.5B-Instruct`, 4-bit) only over your 299 real training rows and,
later, the test rows — never over any external corpus, so this stays cheap regardless of how big the
synthetic corpus above is. The prompt asks a direct yes/no faithfulness question in Bangla/English mixed
instruction and we parse the first token of the reply into a probability via the model's own logits over
the two candidate first tokens, rather than just string-matching generated text (more stable, avoids
depending on exact wording of a free-form generation).

In [ ]:
if ENABLE_QWEN:
    from transformers import BitsAndBytesConfig
    qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL)
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    qwen_model = AutoModelForSequenceClassification.from_pretrained  # placeholder, replaced below
    from transformers import AutoModelForCausalLM
    qwen_model = AutoModelForCausalLM.from_pretrained(
        QWEN_MODEL, quantization_config=bnb_config, device_map="auto"
    )
    qwen_model.eval()

    YES_TOKS = qwen_tokenizer.encode("Yes", add_special_tokens=False)
    NO_TOKS = qwen_tokenizer.encode("No", add_special_tokens=False)
    YES_ID, NO_ID = YES_TOKS[0], NO_TOKS[0]

    def build_prompt(context, question, response):
        ctx_part = f"Context: {context}\n" if context else ""
        return (
            "You are a strict fact-checker for Bangla question answering. "
            "Answer with exactly one word: Yes or No.\n"
            f"{ctx_part}Question: {question}\n"
            f"Proposed answer: {response}\n"
            "Is the proposed answer fully supported by the context (or, if there is no context, is it a "
            "plausible correct answer with no fabricated details)? Answer Yes or No:"
        )

    @torch.no_grad()
    def get_qwen_verdicts(data_df, batch_size=8):
        verdicts = np.full(len(data_df), 0.5)
        prompts = [
            build_prompt(c, q, r)
            for c, q, r in zip(data_df['context'], data_df['prompt_bn'], data_df['response_bn'])
        ]
        for start in tqdm(range(0, len(prompts), batch_size), desc='Qwen verifier'):
            batch_prompts = prompts[start:start + batch_size]
            msgs = [[{"role": "user", "content": p}] for p in batch_prompts]
            texts = [qwen_tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in msgs]
            enc = qwen_tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=768).to(qwen_model.device)
            out = qwen_model(**enc)
            last_logits = out.logits[:, -1, :]
            yes_no_logits = torch.stack([last_logits[:, YES_ID], last_logits[:, NO_ID]], dim=-1)
            probs = F.softmax(yes_no_logits, dim=-1)[:, 0]  # P(Yes) = P(faithful)
            verdicts[start:start + len(batch_prompts)] = probs.cpu().numpy()
        return verdicts

    print("Computing Qwen verifier scores for training data (299 rows only, cheap)...")
    qwen_oof_proxy = get_qwen_verdicts(df)
    df['qwen_verdict'] = qwen_oof_proxy
    print(f"Qwen verdict: mean={qwen_oof_proxy.mean():.3f}")
    qwen_acc = accuracy_score(y, (qwen_oof_proxy >= 0.5).astype(int))
    print(f"Qwen-only accuracy on training rows (zero-shot, no fine-tune — sanity check only): {qwen_acc:.4f}")

    del qwen_model
    gc.collect(); torch.cuda.empty_cache() if device == 'cuda' else None
else:
    df['qwen_verdict'] = 0.5
    print("Qwen verifier disabled — feature set to neutral 0.5.")

## 7. Assemble Full Feature Matrix

In [ ]:
feature_cols = (
    list(hc_train.columns) +
    list(ner_train.columns) +
    ['nli_entail', 'nli_neutral', 'nli_contra', 'nli2_entail', 'nli2_contra',
     'sim_ctx_resp', 'sim_prompt_resp', 'banglabert_oof', 'qwen_verdict']
)

X_full = np.column_stack([
    hc_train.values,
    ner_train.values,
    df['nli_entail'].values.reshape(-1, 1),
    df['nli_neutral'].values.reshape(-1, 1),
    df['nli_contra'].values.reshape(-1, 1),
    df['nli2_entail'].values.reshape(-1, 1),
    df['nli2_contra'].values.reshape(-1, 1),
    df['sim_ctx_resp'].values.reshape(-1, 1),
    df['sim_prompt_resp'].values.reshape(-1, 1),
    df['banglabert_oof'].values.reshape(-1, 1),
    df['qwen_verdict'].values.reshape(-1, 1),
])

print(f"Feature matrix: {X_full.shape}")
print(f"Features: {len(feature_cols)} columns -> {feature_cols}")
print(f"Label distribution: 0={(y==0).sum()}, 1={(y==1).sum()}")

## 8. 5-Fold CV: Logistic Regression + XGBoost (unchanged procedure from v6)

In [ ]:
oof_lr = np.zeros(len(df))
oof_xgb = np.zeros(len(df))
n0, n1 = (y == 0).sum(), (y == 1).sum()

for fold, (tr_idx, va_idx) in enumerate(fold_splits):
    print(f"\n--- Fold {fold} ---")
    X_tr, X_va = X_full[tr_idx], X_full[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    lr = SKPipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(C=0.5, max_iter=1000, class_weight='balanced',
                                    penalty='l2', random_state=RANDOM_STATE)),
    ])
    lr.fit(X_tr, y_tr)
    oof_lr[va_idx] = lr.predict_proba(X_va)[:, 1]

    xgb_clf = xgb.XGBClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, min_child_weight=5,
        reg_lambda=2.0, scale_pos_weight=n0 / n1, random_state=RANDOM_STATE,
        eval_metric='logloss', early_stopping_rounds=20, verbosity=0
    )
    xgb_clf.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
    oof_xgb[va_idx] = xgb_clf.predict_proba(X_va)[:, 1]

    print(f"  LR   Acc={accuracy_score(y_va, (oof_lr[va_idx]>=0.5).astype(int)):.4f}  "
          f"F1={f1_score(y_va, (oof_lr[va_idx]>=0.5).astype(int), average='macro'):.4f}")
    print(f"  XGB  Acc={accuracy_score(y_va, (oof_xgb[va_idx]>=0.5).astype(int)):.4f}  "
          f"F1={f1_score(y_va, (oof_xgb[va_idx]>=0.5).astype(int), average='macro'):.4f}")

print(f"\n=== LR  OOF: Acc={accuracy_score(y,(oof_lr>=0.5).astype(int)):.4f}  F1={f1_score(y,(oof_lr>=0.5).astype(int),average='macro'):.4f}")
print(f"=== XGB OOF: Acc={accuracy_score(y,(oof_xgb>=0.5).astype(int)):.4f}  F1={f1_score(y,(oof_xgb>=0.5).astype(int),average='macro'):.4f}")
print(f"=== BanglaBERT OOF: Acc={accuracy_score(y,(banglabert_oof>=0.5).astype(int)):.4f}  F1={f1_score(y,(banglabert_oof>=0.5).astype(int),average='macro'):.4f}")

## 9. Blend weight search helper

Now a 7-component grid search: LR, XGB, primary NLI-entail, second NLI-entail, embedding-sim, BanglaBERT,
Qwen verdict. The recursive grid search from v6 is unchanged in logic; it just runs over more components,
which is why the coarser `step` below matters more for runtime than it did with 5.

In [ ]:
nli_raw = df['nli_entail'].values
nli2_raw = df['nli2_entail'].values
sim_raw = df['sim_ctx_resp'].values
bb_raw = df['banglabert_oof'].values
qwen_raw = df['qwen_verdict'].values

OOF_COMPONENTS = {
    'lr': oof_lr, 'xgb': oof_xgb, 'nli': nli_raw, 'nli2': nli2_raw,
    'sim': sim_raw, 'bb': bb_raw, 'qwen': qwen_raw,
}

def grid_search_blend(oof_components, y_subset, idx_subset, step=0.1, thresholds=np.arange(0.30, 0.71, 0.02)):
    """Grid search blend weights (summing to 1) and threshold on the given subset of indices only.
    step widened to 0.1 (from 0.05 in v6) since we now have 7 components instead of 5 — keeps the
    combinatorial grid size from exploding. Re-tighten to 0.05 if you have time budget to spare."""
    names = list(oof_components.keys())
    arrs = [oof_components[n][idx_subset] for n in names]
    y_sub = y_subset

    best_f1, best_w, best_t = -1, None, 0.5
    grid_vals = np.arange(0, 1.01, step)
    def rec(k, remaining, chosen):
        nonlocal best_f1, best_w, best_t
        if k == len(names) - 1:
            w = chosen + [remaining]
            if remaining < -1e-6:
                return
            blend = sum(wi * ai for wi, ai in zip(w, arrs))
            for t in thresholds:
                p = (blend >= t).astype(int)
                f1 = f1_score(y_sub, p, average='macro')
                if f1 > best_f1:
                    best_f1, best_w, best_t = f1, tuple(w), t
            return
        for wv in grid_vals:
            if wv > remaining + 1e-6:
                break
            rec(k + 1, round(remaining - wv, 4), chosen + [wv])
    rec(0, 1.0, [])
    return dict(zip(names, best_w)), best_t, best_f1

## 10. Honest Nested-CV Score (this is the number to trust)

Unchanged methodology from v6: for each outer fold, blend weights and threshold are selected using only
the other 4 folds, then applied — unseen — to the held-out fold.

In [ ]:
outer_fold_scores = []

for fold, (tr_idx, va_idx) in enumerate(fold_splits):
    weights, thresh, inner_f1 = grid_search_blend(OOF_COMPONENTS, y[tr_idx], tr_idx)
    blend_va = sum(w * OOF_COMPONENTS[name][va_idx] for name, w in weights.items())
    preds_va = (blend_va >= thresh).astype(int)
    fold_f1 = f1_score(y[va_idx], preds_va, average='macro')
    outer_fold_scores.append(fold_f1)
    print(f"Outer fold {fold}: inner-selected weights={ {k: round(v,2) for k,v in weights.items()} }  "
          f"thresh={thresh:.2f}  ->  held-out MacroF1={fold_f1:.4f}")

honest_cv_score = np.mean(outer_fold_scores)
honest_cv_std = np.std(outer_fold_scores)
print(f"\n=== Honest nested-CV Macro-F1: {honest_cv_score:.4f} (+/- {honest_cv_std:.4f}) ===")
print("This is the number that should predict your leaderboard score, not any of the earlier ones.")
print("Compare this against your v6 honest score (0.64-ish baseline) to see whether the synthetic")
print("pretrain + second NLI + Qwen verifier actually helped, before trusting the deployment blend below.")

## 11. Deployment Blend (for generating the actual submission)

Full-data grid search for the weights/threshold actually used at inference. This number will look
better than Section 10 — expected, and not the number to report to yourself as "my score."

In [ ]:
all_idx = np.arange(len(df))
deploy_weights, deploy_thresh, deploy_f1 = grid_search_blend(OOF_COMPONENTS, y, all_idx)
print(f"Deployment weights: { {k: round(v,2) for k,v in deploy_weights.items()} }")
print(f"Deployment threshold: {deploy_thresh:.2f}")
print(f"(In-sample OOF Macro-F1 with these weights: {deploy_f1:.4f} — optimistic, see Section 10 for the honest estimate)")

final_blend = sum(w * OOF_COMPONENTS[name] for name, w in deploy_weights.items())
final_preds = (final_blend >= deploy_thresh).astype(int)
print(confusion_matrix(y, final_preds))
print(classification_report(y, final_preds, target_names=['hallucinated(0)', 'faithful(1)']))

## 12. Test Set Inference

In [ ]:
if os.path.exists(TEST_PATH):
    test_df = pd.read_csv(TEST_PATH)
    print(f"Test: {len(test_df)}")

    for col in ['context', 'prompt_bn', 'response_bn']:
        test_df[col] = test_df[col].apply(clean)
    test_df['has_context'] = test_df['context'].str.len() > 0
    test_df['premise'] = test_df['context'].where(test_df['has_context'], test_df['prompt_bn'])

    # --- BanglaBERT: average predictions across the 5 fold models ---
    if ENABLE_BANGLABERT:
        test_bb_probs = np.zeros(len(test_df))
        for model, tokenizer in banglabert_fold_models:
            model.eval()
            ds = PairDataset(test_df['premise'], test_df['response_bn'], None, tokenizer)
            dl = DataLoader(ds, batch_size=16, shuffle=False)
            fold_probs = []
            with torch.no_grad():
                for batch in dl:
                    batch = {k: v.to(device) for k, v in batch.items()}
                    out = model(**batch)
                    fold_probs.extend(F.softmax(out.logits, dim=-1)[:, 1].cpu().numpy().tolist())
            test_bb_probs += np.array(fold_probs) / len(banglabert_fold_models)
        test_df['banglabert_oof'] = test_bb_probs
    else:
        test_df['banglabert_oof'] = 0.5

    # --- Primary NLI ---
    print("Computing test primary NLI scores...")
    nli_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
    nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(device)
    nli_model.eval()
    test_nli_entail, test_nli_neutral, test_nli_contra = get_nli_scores(
        test_df, nli_tokenizer, nli_model, NLI_ENTAIL_IDX, NLI_NEUTRAL_IDX, NLI_CONTRA_IDX
    )
    test_df['nli_entail'] = test_nli_entail
    test_df['nli_neutral'] = test_nli_neutral
    test_df['nli_contra'] = test_nli_contra
    del nli_model; gc.collect(); torch.cuda.empty_cache() if device == 'cuda' else None

    # --- Second NLI ---
    if ENABLE_NLI2:
        print("Computing test second-NLI scores...")
        nli2_tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL_2)
        nli2_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_2).to(device)
        nli2_model.eval()
        test_nli2_entail, test_nli2_neutral, test_nli2_contra = get_nli_scores(
            test_df, nli2_tokenizer, nli2_model, NLI2_ENTAIL_IDX, NLI2_NEUTRAL_IDX, NLI2_CONTRA_IDX
        )
        test_df['nli2_entail'] = test_nli2_entail
        test_df['nli2_contra'] = test_nli2_contra
        del nli2_model; gc.collect(); torch.cuda.empty_cache() if device == 'cuda' else None
    else:
        test_df['nli2_entail'] = 1/3
        test_df['nli2_contra'] = 1/3

    # --- Embeddings ---
    print("Computing test embedding similarity...")
    emb_model = SentenceTransformer(EMB_MODEL, device=str(device))
    test_ctx_texts = test_df['premise'].tolist()
    test_resp_texts = test_df['response_bn'].tolist()
    test_prompt_texts = test_df['prompt_bn'].tolist()
    test_sim_ctx_resp, _, _ = cosine_sim_matrix(test_ctx_texts, test_resp_texts)
    test_sim_prompt_resp, _, _ = cosine_sim_matrix(test_prompt_texts, test_resp_texts)
    test_df['sim_ctx_resp'] = test_sim_ctx_resp
    test_df['sim_prompt_resp'] = test_sim_prompt_resp
    del emb_model; gc.collect(); torch.cuda.empty_cache() if device == 'cuda' else None

    # --- NER + hand-crafted ---
    print("Computing test NER + hand-crafted features...")
    ner_test = get_ner_features(test_df)
    hc_test = build_features(test_df)

    # --- Qwen verifier ---
    if ENABLE_QWEN:
        print("Computing test Qwen verifier scores...")
        from transformers import BitsAndBytesConfig, AutoModelForCausalLM
        qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL)
        bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
        qwen_model = AutoModelForCausalLM.from_pretrained(QWEN_MODEL, quantization_config=bnb_config, device_map="auto")
        qwen_model.eval()
        test_qwen = get_qwen_verdicts(test_df)
        test_df['qwen_verdict'] = test_qwen
        del qwen_model; gc.collect(); torch.cuda.empty_cache() if device == 'cuda' else None
    else:
        test_df['qwen_verdict'] = 0.5

    X_test = np.column_stack([
        hc_test.values,
        ner_test.values,
        test_df['nli_entail'].values.reshape(-1, 1),
        test_df['nli_neutral'].values.reshape(-1, 1),
        test_df['nli_contra'].values.reshape(-1, 1),
        test_df['nli2_entail'].values.reshape(-1, 1),
        test_df['nli2_contra'].values.reshape(-1, 1),
        test_df['sim_ctx_resp'].values.reshape(-1, 1),
        test_df['sim_prompt_resp'].values.reshape(-1, 1),
        test_df['banglabert_oof'].values.reshape(-1, 1),
        test_df['qwen_verdict'].values.reshape(-1, 1),
    ])
    print(f"Test features: {X_test.shape}")

    # --- Retrain LR / XGB on full training data for test-time inference ---
    lr_full = SKPipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(C=0.5, max_iter=1000, class_weight='balanced',
                                    penalty='l2', random_state=RANDOM_STATE)),
    ])
    lr_full.fit(X_full, y)
    lr_test_prob = lr_full.predict_proba(X_test)[:, 1]

    xgb_full = xgb.XGBClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.7, min_child_weight=5,
        reg_lambda=2.0, scale_pos_weight=n0 / n1, random_state=RANDOM_STATE,
        eval_metric='logloss', verbosity=0
    )
    xgb_full.fit(X_full, y)
    xgb_test_prob = xgb_full.predict_proba(X_test)[:, 1]

    test_components = {
        'lr': lr_test_prob, 'xgb': xgb_test_prob,
        'nli': test_nli_entail, 'nli2': test_nli2_entail,
        'sim': test_sim_ctx_resp, 'bb': test_df['banglabert_oof'].values,
        'qwen': test_df['qwen_verdict'].values,
    }
    test_blend = sum(deploy_weights[name] * arr for name, arr in test_components.items())
    test_preds = (test_blend >= deploy_thresh).astype(int)

    n0_pred, n1_pred = (test_preds == 0).sum(), (test_preds == 1).sum()
    print(f"Test distribution: 0={n0_pred} ({n0_pred/len(test_preds)*100:.1f}%), "
          f"1={n1_pred} ({n1_pred/len(test_preds)*100:.1f}%)")

    id_col = test_df['id'] if 'id' in test_df.columns else range(len(test_df))
    sub = pd.DataFrame({'id': id_col, 'label': test_preds})
    os.makedirs(os.path.dirname(SUBMISSION_PATH), exist_ok=True)
    sub.to_csv(SUBMISSION_PATH, index=False)
    print(f"Saved: {SUBMISSION_PATH}")
    sub.head(10)
else:
    print("Test set CSV file not found. Skipping prediction. Ready for Kaggle run.")

## Summary

**v7.0 changes vs v6.0:**
- Added a **synthetic, task-shaped pretraining corpus** (Section 1b) built from BanglaRQA and the
  rasheduzzaman Bangla QA-pair dataset — real `(context, question, answer)` triples turned into balanced
  faithful/hallucinated pairs via four corruption strategies. Explicitly did **not** use the generic
  Alpaca-Bangla instruction datasets, since they carry no faithfulness label and don't match this task's
  shape.
- **Two-stage BanglaBERT training**: Stage 1 pretrains on the synthetic corpus, Stage 2 fine-tunes on the
  real 299 rows via the same 5-fold CV as v6, now warm-started from the Stage-1 checkpoint.
- Added a **second, independent NLI checkpoint** as its own blend component (not merged with the first).
- Added an **optional Qwen2.5-1.5B-Instruct verifier feature**, run only on the 299 real rows + test set
  (never on the synthetic corpus), gated by `ENABLE_QWEN` for time-budget control.
- Blend search widened from 5 to 7 components; kept the honest nested-CV methodology from v6 unchanged —
  Section 10 is still the number to trust, Section 11 is still just for building the submission.

**What to check when you run this:**
- Compare Section 10's `honest_cv_score` here against your v6 honest score. If it isn't meaningfully
  higher, the synthetic corpus's distribution likely doesn't match the true hallucination patterns in your
  actual test set well enough — that's useful information, not a failure, and tells you to either refine
  the corruption strategies or lean more on the hand-crafted/NLI features instead.
- If you're tight on the 9-hour budget, disable `ENABLE_QWEN` first (most expensive new piece), then
  `ENABLE_SYN_PRETRAIN` if still tight — both degrade gracefully to v6 behavior when off.
- With ~299 real rows, per-fold variance in Section 10 will still be non-trivial; report the standard
  deviation alongside the mean, not just the mean.
